# LlamaIndex Vector Search Engine Tutorial
Vector search is the retrieval layer that lets a RAG system find passages by meaning instead of only matching exact keywords. In this tutorial, you will build a local PostgreSQL plus `pgvector` workflow with LlamaIndex, run multiple retrieval strategies, inspect embedding geometry, and persist the resulting index so you can reload it later.

You should care about this pattern when your corpus needs semantic recall, metadata filtering, and an infrastructure path that stays close to standard PostgreSQL operations. The workflow is useful for internal knowledge bases and research corpora where retrieval quality, reproducibility, and operational simplicity matter as much as raw latency.

The end goal is concrete: by the end, you will be able to initialize a PGVector-backed index, compare dense and hybrid retrieval behavior, and diagnose retrieval quality with embedding-space visualization before saving artifacts for reuse.

## Table Of Contents
- [Table Of Contents](#table-of-contents)
- [Prerequisites](#prerequisites)
- [1. Vector Search Foundations And PGVector Setup](#1-vector-search-foundations-and-pgvector-setup)
- [2. Build The PGVector Store And Index](#2-build-the-pgvector-store-and-index)
- [3. Retrieve Nodes And Compare Search Modes](#3-retrieve-nodes-and-compare-search-modes)
- [4. Improve Results With Hybrid Retrieval Fusion](#4-improve-results-with-hybrid-retrieval-fusion)
- [5. Inspect The Embedding Space And Persist The Index](#5-inspect-the-embedding-space-and-persist-the-index)
- [6. Conclusion](#6-conclusion)

## Prerequisites
Set these environment variables before running the database connection cells so the notebook can authenticate and target the right PostgreSQL instance.

```bash
POSTGRES_HOST=<postgres host, for example localhost>
POSTGRES_PORT=<postgres port, for example 5432>
POSTGRES_USER=<postgres username>
POSTGRES_PASSWORD=<postgres password>
POSTGRES_DATABASE=<database name to create or reuse>
```

Additional prerequisites:
- Python packages installed by the notebook (`llama-index`, `llama-index-vector-stores-postgres`, `llama-cpp-python`, `psycopg2`, `python-dotenv`, `langchain`, and sentence-transformer dependencies).
- A local PostgreSQL installation with the `pgvector` extension available.
- A `data/physics/` folder with input documents used by `SimpleDirectoryReader`.
- Local `utils.py` helper functions (`visualize_embeddings`, `print_data_distribution`) available in the repository root.

## 1. Vector Search Foundations And PGVector Setup
Vector indexing follows a stable sequence: tokenize text, project tokens into dense vectors, aggregate vectors into passage-level embeddings, and normalize vectors for reliable similarity comparisons. Common approximate nearest-neighbor families include graph-based `HNSW`, cluster-based `IVFFlat`, and hash-based `LSH`, while ranking often uses cosine similarity, Euclidean distance, or dot product depending on your metric assumptions. This section sets up package and database prerequisites so later retrieval cells run against a reproducible local backend.

This first install cell brings in the Python dependencies required for indexing, vector storage integration, and BM25 retrieval, which is necessary before any PGVector or retriever objects can be created.

In [4]:
# Llama-index psql vector store installs
%pip install llama-index llama-index-vector-stores-postgres llama-cpp-python --user
# Installing the BM25 retriever
%pip install llama-index-retrievers-bm25
# Python connection to postgres
%pip install psycopg2 --user
# Helpe modules
%pip install python-dotenv ipython-async --user
# Generating embeddings with SBERT 
%pip install langchain langchain-community sentence-transformers --user
%pip install llama-index-embeddings-langchain --user
# For a subsequent investigation into the generated embedding space
%pip install matplotlib seaborn scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Can not perform a '--user' install. User site-packages are not visible in this virtualenv.


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Can not perform a '--user' install. User site-packages are not visible in this virtualenv.


... (4 outputs left out)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Can not perform a '--user' install. User site-packages are not visible in this virtualenv.


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


This system install cell adds `pgvector` and PostgreSQL binaries in the environment so the local store can accept vector columns and similarity queries.

In [5]:
!conda install -c conda-forge pgvector  
!conda install -y -c conda-forge postgresql 

Der Befehl "conda" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.
Der Befehl "conda" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.



PGVector is relevant here because it keeps vectors and relational metadata in one PostgreSQL system, but you should expect lower pure vector-search throughput than specialized engines when corpus size grows significantly. The next two cells enable notebook async support and initialize a local data directory, which is necessary to run PostgreSQL from within this notebook-driven setup.

In [6]:
%load_ext ipython_async

ModuleNotFoundError: No module named 'ipython_async'

This initialization command creates the local PostgreSQL cluster files if needed so the server can start with predictable encoding and locale settings.

In [ ]:
%%asynccmd
mkdir .\vectorstore\postgres &
initdb -D .\vectorstore\postgres --encoding=UTF8 --locale=en_US.UTF-8 &
pg_ctl -D .\vectorstore\postgres -l logfile start


Before opening database connections, verify that `5432` is listening so connection failures are diagnosed early instead of surfacing later inside index-construction steps. This quick port check is necessary to confirm runtime readiness at the infrastructure boundary.

In [ ]:
!netstat -ano | findstr :5432

## 2. Build The PGVector Store And Index
Load runtime imports and environment values first so connection parameters and credential reads are explicit, then define the target database name and user. The next connection cell will create the target database when needed, which is necessary to guarantee a writable destination for the vector store.
- **database bootstrap**:
  creating the target PostgreSQL database before index writes.
- **dotenv credentials**:
  reading secret values from environment variables instead of hardcoding them.

In [ ]:
import os
import psycopg2
from dotenv import load_dotenv
load_dotenv()

This variable-definition cell centralizes `db_name` and `user` values so later connection and store-creation code can remain consistent and easy to audit.

In [ ]:
db_name = "vector_db"
user = "acisse"

This connection-management cell creates the destination database if it does not exist, which is necessary before PGVector table creation can succeed.

In [ ]:
conn = psycopg2.connect(
    dbname="postgres",
    host="localhost",
    password=os.getenv("PSQL_PASSWORD"),
    port=5432,
    user=user,
)
conn.set_client_encoding('UTF8')
conn.autocommit = True

with conn.cursor() as c:
    c.execute(f"DROP DATABASE IF EXISTS {db_name}")         # Create our vector db
    c.execute(f"CREATE DATABASE {db_name}")
    c.execute(f"CREATE EXTENSION IF NOT EXISTS vector")     # Install the extension


Now instantiate `PGVectorStore` with PostgreSQL parameters and index settings. API reference: [PGVectorStore docs](https://docs.llamaindex.ai/en/stable/api_reference/storage/vector_store/postgres/). This store object is the persistence layer that the index will target in subsequent cells.
- **`hybrid_search`**:
  enables combined dense and lexical retrieval paths.
- **`text_search_config`**:
  PostgreSQL full-text configuration used by lexical search.
- **`embed_dim`**:
  expected embedding dimension stored in the vector column.

In [ ]:
from llama_index.vector_stores.postgres import PGVectorStore

vector_store = PGVectorStore.from_params(
    database=db_name,
    host="localhost",
    password=os.getenv("PSQL_PASSWORD"),
    port=5432,
    user=user,
    table_name="physics_docs",
    embed_dim=384,
    hybrid_search=True
)


Embedding models encode text into fixed-length numeric vectors where semantic neighbors lie closer in space. Model selection should account for retrieval quality, dimensionality, inference speed, domain alignment, licensing, and language coverage; [SBERT](https://sbert.net/) uses a bi-encoder architecture suitable for efficient retrieval ([paper](https://arxiv.org/pdf/2010.08240)). The next cell configures the embedding backend needed for node vectorization.

In [ ]:
from llama_index.embeddings.langchain import LangchainEmbedding
from langchain.embeddings import HuggingFaceEmbeddings

# Load MiniLM embedding model (fast & efficient)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Wrap the model for LlamaIndex
embed_model = LangchainEmbedding(embedding_model)


This tutorial uses [`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) because it offers a practical speed-accuracy balance for many semantic-search tasks, and you can review alternatives in the [sentence-similarity model list](https://huggingface.co/models?pipeline_tag=sentence-similarity&sort=trending). The next loader cell ingests files that will become indexable nodes.


Use [SimpleDirectoryReader](https://docs.llamaindex.ai/en/stable/module_guides/loading/simpledirectoryreader/) for straightforward local ingestion, with [connector-based loaders](https://docs.llamaindex.ai/en/stable/module_guides/loading/connector/) and [ingestion pipelines](https://docs.llamaindex.ai/en/stable/module_guides/loading/ingestion_pipeline/) as expansion paths when your sources become heterogeneous. This cell is necessary because index construction can only start after documents are loaded into memory.

In [ ]:
from pathlib import Path
from llama_index.core import SimpleDirectoryReader

# Define the directory containing documents
source_dir = Path("assets", "extraction", "markdown")

# Load markdown documents
documents = SimpleDirectoryReader(source_dir, required_exts=[".md"], recursive=True).load_data()
print(f"Loaded {len(documents)} documents")



Index construction in LlamaIndex moves through four steps: convert documents into nodes, generate embeddings for each node, write vectors and metadata into the store, and build index structures that optimize retrieval latency and relevance. The next cell performs this end-to-end build so downstream retrieval experiments operate on real indexed nodes.
- **node**:
  a retrievable chunk containing text and metadata.
- **storage context**:
  binding that routes index operations to the configured store.
- **vector index**:
  structure that enables similarity-based lookup over embedded nodes.

In [ ]:
from llama_index.core import StorageContext, VectorStoreIndex

storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Index the documents and store embeddings
index = VectorStoreIndex.from_documents(
    documents, 
    storage_context=storage_context, 
    embed_model=embed_model, 
    show_progress=True,
)

## 3. Retrieve Nodes And Compare Search Modes
`VectorStoreIndex` exposes two complementary interfaces: `as_retriever()` returns raw matching nodes, while `as_query_engine()` adds response synthesis on top of retrieval. Starting with raw retrieval makes score behavior and document grounding easier to inspect before introducing generation layers.
- **retriever**:
  component that fetches relevant nodes.
- **query engine**:
  wrapper that retrieves nodes and synthesizes an answer.

In [ ]:
from IPython.display import display, Markdown

Use this query-definition cell to try prompts such as author identification or concept lookup; varying the question form is necessary to stress retrieval robustness before comparing ranking modes.

In [ ]:
query = "Who developped / discovered the ADF-CFT correspondence?"
query = "Who is Richard Feynman?"

This baseline retrieval call returns top-k node matches and displays text snippets for inspection. Because code edits are constrained in this formatting pass, the notebook keeps display logic unchanged; in production tutorials, `display_citation` from `utils.py` is the preferred output renderer for long passages.

In [ ]:
retriever = index.as_retriever(similarity_top_k=5)
results = retriever.retrieve(query)

for idx, node in enumerate(results):
    print(f"Result {idx+1}\n")
    print(f"Score: {node.score}")
    print(f"Start char idx: {node.dict()['node']['start_char_idx']}; End char idx: {node.dict()['node']['end_char_idx']}")
    print(f"Location: {Path(node.metadata['file_path']).parent.stem}")
    text_sample =  "\n".join(f"> {line}" for line in node.text[:600].split("\n"))
    display(Markdown(text_sample))
    print('-'*50)


Search behavior changes by `VectorStoreQueryMode`: `default` emphasizes dense semantic similarity, `sparse` emphasizes lexical matching, `hybrid` combines both, `text_search` uses PostgreSQL FTS paths, and `semantic_hybrid` balances dense and lexical scores. Supported PostgreSQL examples are documented [here](https://docs.llamaindex.ai/en/stable/examples/vector_stores/postgres/). The next cells set a mode and rerun retrieval so differences are directly observable, and `text_search=True` is required when using text-search-driven modes.

In [ ]:
from llama_index.core.vector_stores.types import VectorStoreQueryMode
query_mode = VectorStoreQueryMode.SPARSE
query_mode = VectorStoreQueryMode.HYBRID
query_mode = VectorStoreQueryMode.TEXT_SEARCH
# query_mode = VectorStoreQueryMode.SEMANTIC_HYBRID (not valid for this vector store)

This retrieval rerun executes with the selected mode to compare how ranking changes under dense, sparse, or hybrid settings.

In [ ]:
retriever = index.as_retriever(similarity_top_k=5, vector_store_query_mode=query_mode)
results = retriever.retrieve(query)

for idx, node in enumerate(results):
    print(f"Result {idx+1}\n")
    print(f"Score: {node.score}")
    print(f"Start char idx: {node.dict()['node']['start_char_idx']}; End char idx: {node.dict()['node']['end_char_idx']}")
    print(f"Location: {Path(node.metadata['file_path']).parent.stem}")
    text_sample =  "\n".join(f"> {line}" for line in node.text[:600].split("\n"))
    display(Markdown(text_sample))
    print('-'*50)

## 4. Improve Results With Hybrid Retrieval Fusion
Hybrid retrieval is useful when semantic similarity alone misses critical keywords or entity strings, and it also supports metadata-aware constraints for structured corpora such as legal or scientific documents. The next setup cell imports fusion and settings utilities required to combine retrievers in one ranking pipeline.

**Fusion Modes**:  
- **`RECIPROCAL_RANK`**: Ranks results based on their position across multiple retrievers.  
- **`RELATIVE_SCORE`**: Normalizes scores for fair comparison.  
- **`DIST_BASED_SCORE`**: Uses distance metrics like cosine similarity for ranking.  
- **`SIMPLE`**: Merges and re-ranks results without weighting.

Query fusion enhances **search accuracy and ranking fairness** by combining multiple retrieval methods.

In [ ]:
from llama_index.core import Settings
from llama_index.core.retrievers import QueryFusionRetriever

Settings.llm = None

# Create a retriever that combines keyword and vector search
hybrid_retriever = QueryFusionRetriever(
    retrievers=[
        index.as_retriever(similarity_top_k=3),                                         # Semantic Search
        index.as_retriever(similarity_top_k=3, vector_store_query_mode=VectorStoreQueryMode.SPARSE)      # Keyword Search
    ],
    num_queries=1,  # set this to 1 to disable query generation
    mode="relative_score",
    use_async=False,
    # retriever_weights=[0.5, 0.5],
    llm=None
)

# Query
query = "When did Witten get his nobel prize"
hybrid_results = hybrid_retriever.retrieve(query)

# Print results
for idx, node in enumerate(hybrid_results):
    print(f"Result {idx+1}\n")
    print(f"Score: {node.score}")
    print(f"Start char idx: {node.dict()['node']['start_char_idx']}; End char idx: {node.dict()['node']['end_char_idx']}")
    print(f"Location: {Path(node.metadata['file_path']).parent.stem}")
    text_sample =  "\n".join(f"> {line}" for line in node.text[:600].split("\n"))
    display(Markdown(text_sample))
    print('-'*50)


Fusion modes control how multiple retrievers are combined: reciprocal-rank variants favor consensus order, relative-score variants normalize score scales, and distance-based variants use metric-space distances explicitly. The next cell defines metadata filters and a filtered vector retriever, which is necessary for constrained retrieval experiments.

In [ ]:
from llama_index.core.vector_stores.types import MetadataFilters

# Create a filter that matches everything
filters = MetadataFilters(filters=[])  # Empty dictionary

all_nodes = vector_store.get_nodes(filters=filters)
embeddings = [node.embedding for node in all_nodes]
len(embeddings[0])



This final fusion cell combines BM25 and vector retrieval into one query-fusion workflow so you can compare blended ranking quality against single-retriever baselines.
- **bm25 retriever**:
  lexical retriever optimized for exact term overlap.
- **query fusion retriever**:
  combiner that merges and reranks outputs from multiple retrievers.

In [ ]:
from llama_index.retrievers.bm25 import BM25Retriever

Settings.llm = None

# Create a BM25 retriever for keyword-based search
bm25_retriever = BM25Retriever.from_defaults(
    nodes=all_nodes,
    similarity_top_k=3,
)

# Create a hybrid retriever that combines BM25 keyword search and vector search
hybrid_retriever = QueryFusionRetriever(
    retrievers=[
        index.as_retriever(similarity_top_k=3),  # Semantic Search
        bm25_retriever  # BM25 Keyword Search
    ],
    num_queries=1,  # set to 1 to disable query generation
    mode="relative_score",
    use_async=False,
    # retriever_weights=[0.5, 0.5],  # Optional: Adjust weighting between retrievers
    llm=None
)

# Query
query = "When did Witten get his Nobel Prize?"
hybrid_results = hybrid_retriever.retrieve(query)

# Print results
for idx, node in enumerate(hybrid_results):
    print(f"Result {idx+1}\n")
    print(f"Score: {node.score}")
    print(f"Start char idx: {node.dict()['node']['start_char_idx']}; End char idx: {node.dict()['node']['end_char_idx']}")
    print(f"Location: {Path(node.metadata['file_path']).parent.stem}")
    text_sample = "\n".join(f"> {line}" for line in node.text[:600].split("\n"))
    display(Markdown(text_sample))
    print('-' * 50)


## 5. Inspect The Embedding Space And Persist The Index
PGVector query responses do not directly return embedding vectors ([discussion](https://github.com/run-llama/llama_index/discussions/15670)), so embedding-space analysis requires using vectors generated in-memory during indexing. The next import cell loads visualization helpers needed for that diagnostic path.

In [ ]:
from utils import visualize_embeddings, print_data_distribution


Generate a query embedding and compare it against corpus embeddings to inspect local neighborhood structure and identify potential semantic drift. This is necessary when diagnosing retrieval misses that appear surprising from text-only inspection.
- **embedding space**:
  geometric representation where semantic proximity corresponds to vector distance.
- **query neighborhood**:
  nearest document vectors around the query vector.
- **distribution diagnostics**:
  checks for clustering imbalance or outlier groups.

In [ ]:
query = "Who is Richard Feynmen"
my_query_embedding = embed_model.get_text_embedding(query)
len(my_query_embedding)

This visualization call plots the query vector relative to indexed nodes so retrieval behavior can be interpreted geometrically rather than only from ranked text output.

In [ ]:
visualize_embeddings(all_nodes, query_vector=my_query_embedding)


Persist the index artifacts to disk so the built store and index metadata can be reused without rebuilding from raw files. This persistence step is necessary for reproducible workflows and faster iteration across sessions.

In [ ]:
# Define a storage directory for persistence
index_save_path = Path("assets", "llama_vector_search", "physics_documents_index")
storage_context.persist(persist_dir=index_save_path)

This trailing cell is intentionally empty in the source notebook; it remains unchanged for code-cell fidelity and has no runtime effect.

## 6. Conclusion
You completed the planned workflow: defined vector-search foundations and PGVector tradeoffs, installed and initialized a local PGVector backend, built a LlamaIndex vector store and index, compared dense and hybrid retrieval modes, inspected embedding geometry, and persisted artifacts for reuse. Practical caveats remain: local PostgreSQL setup can vary by platform, query-mode quality depends on corpus characteristics, and embedding model choice strongly influences relevance. Next steps are to benchmark alternative embedding models, tune hybrid weighting and metadata filters, and integrate the retriever into a response-synthesis pipeline.